# 📊 Notebook 01 — Exploratory Data Analysis
## BI Predictive Analytics Platform · Customer Churn Intelligence

---

> **Author:** Mohammed Farhan Khan  
> **Platform:** [bi-predictive-analytics-platform](https://github.com/Farhan786-Khan/bi-predictive-analytics-platform)  
> **Stack:** Python · Pandas · Plotly · Seaborn

---

## 🎯 Business Problem

Customer churn — when a customer stops doing business with a company — is one of the **most expensive problems in enterprise**. Studies show:

- It costs **5–25× more** to acquire a new customer than retain an existing one
- A **5% reduction in churn** can increase profits by **25–95%** (Harvard Business Review)
- Telecom companies lose **$62 billion/year** to churn (CallMiner, 2020)

This notebook explores the customer dataset to understand **who churns, why, and when** — the foundation for everything that follows.

---

## 📋 Table of Contents

1. [Setup & Data Loading](#1-setup)
2. [Dataset Overview](#2-overview)
3. [Target Variable Analysis](#3-target)
4. [Numerical Feature Distributions](#4-numerical)
5. [Categorical Feature Analysis](#5-categorical)
6. [Correlation Analysis](#6-correlation)
7. [Customer Segments](#7-segments)
8. [Key Insights Summary](#8-insights)

## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Consistent dark theme across all plots
DARK = 'plotly_dark'
COLORS = ['#1A56DB', '#EF4444', '#06B6D4', '#22C55E', '#F59E0B', '#7C3AED']
plt.style.use('dark_background')

print('✅ Libraries loaded successfully')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

In [ ]:
from src.data_pipeline.data_loader import DataLoader

loader = DataLoader(data_dir='data')

# Generate reproducible synthetic dataset (or swap in your real data)
df = loader.generate_sample_churn_data(n=5000, seed=42)
df.to_csv('../data/sample_churn_data.csv', index=False)

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Churn rate: {df["churn"].mean():.1%}')
df.head(8)

## 2. Dataset Overview <a id='2-overview'></a>

In [ ]:
print('=' * 60)
print('DATASET SHAPE:  ', df.shape)
print('MEMORY USAGE:   ', f'{df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print('MISSING VALUES: ', df.isna().sum().sum())
print('DUPLICATES:     ', df.duplicated().sum())
print('=' * 60)
print()
print('COLUMN TYPES:')
print(df.dtypes.to_string())

In [ ]:
df.describe().T.style \
    .background_gradient(cmap='Blues', subset=['mean', 'std']) \
    .format(precision=2) \
    .set_caption('📊 Descriptive Statistics — All Numerical Features')

## 3. Target Variable Analysis <a id='3-target'></a>

In [ ]:
churn_counts = df['churn'].value_counts()
churn_labels = {0: 'Retained', 1: 'Churned'}

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'pie'}, {'type': 'bar'}, {'type': 'indicator'}]],
    subplot_titles=['Distribution', 'Count by Segment', 'Churn Rate KPI']
)

# Donut chart
fig.add_trace(go.Pie(
    labels=['Retained', 'Churned'],
    values=[churn_counts[0], churn_counts[1]],
    marker_colors=['#1A56DB', '#EF4444'],
    hole=0.55, textinfo='label+percent',
    textfont_size=13
), row=1, col=1)

# Bar chart
fig.add_trace(go.Bar(
    x=['Retained', 'Churned'],
    y=[churn_counts[0], churn_counts[1]],
    marker_color=['#1A56DB', '#EF4444'],
    text=[churn_counts[0], churn_counts[1]],
    textposition='outside', textfont_size=14
), row=1, col=2)

# KPI gauge
fig.add_trace(go.Indicator(
    mode='gauge+number+delta',
    value=df['churn'].mean() * 100,
    number={'suffix': '%', 'font': {'size': 32, 'color': '#EF4444'}},
    delta={'reference': 15, 'valueformat': '.1f', 'suffix': '% vs 15% benchmark'},
    gauge={
        'axis': {'range': [0, 50]},
        'bar': {'color': '#EF4444'},
        'steps': [
            {'range': [0, 15], 'color': '#16A34A22'},
            {'range': [15, 30], 'color': '#F59E0B22'},
            {'range': [30, 50], 'color': '#EF444422'}
        ],
        'threshold': {'line': {'color': 'white', 'width': 3}, 'value': 15}
    },
    title={'text': 'Churn Rate'}
), row=1, col=3)

fig.update_layout(
    template=DARK, height=380,
    title_text='🎯 Customer Churn — Target Variable Overview',
    title_font=dict(size=18, color='#E2E8F0'),
    showlegend=False
)
fig.show()

## 4. Numerical Feature Distributions <a id='4-numerical'></a>

In [ ]:
num_features = ['tenure_months', 'monthly_charges', 'total_charges', 'support_calls', 'satisfaction_score', 'num_products']

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f.replace('_', ' ').title() for f in num_features]
)

for i, feat in enumerate(num_features):
    row, col = i // 3 + 1, i % 3 + 1
    for churn_val, color, label in [(0, '#1A56DB', 'Retained'), (1, '#EF4444', 'Churned')]:
        subset = df[df['churn'] == churn_val][feat]
        fig.add_trace(go.Violin(
            y=subset, name=label, legendgroup=label,
            showlegend=(i == 0),
            line_color=color, fillcolor=color.replace(')', ', 0.15)').replace('rgb', 'rgba') if 'rgb' in color else color + '26',
            box_visible=True, meanline_visible=True,
            x0=label
        ), row=row, col=col)

fig.update_layout(
    template=DARK, height=550, violinmode='group',
    title_text='📈 Numerical Feature Distributions by Churn Status',
    title_font=dict(size=17, color='#E2E8F0'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

In [ ]:
# Tenure buckets — key churn predictor
df['tenure_bucket'] = pd.cut(df['tenure_months'],
    bins=[0, 12, 24, 36, 48, 72],
    labels=['0-12m', '13-24m', '25-36m', '37-48m', '49-72m'])

tenure_churn = df.groupby('tenure_bucket', observed=True)['churn'].agg(['mean', 'count']).reset_index()
tenure_churn.columns = ['bucket', 'churn_rate', 'count']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Churn Rate by Tenure Bucket', 'Customer Count by Bucket'])

fig.add_trace(go.Bar(
    x=tenure_churn['bucket'].astype(str),
    y=tenure_churn['churn_rate'],
    marker_color=['#EF4444' if r > 0.35 else '#F59E0B' if r > 0.20 else '#22C55E'
                  for r in tenure_churn['churn_rate']],
    text=[f'{r:.1%}' for r in tenure_churn['churn_rate']],
    textposition='outside', name='Churn Rate'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=tenure_churn['bucket'].astype(str), y=tenure_churn['count'],
    marker_color='#1A56DB', text=tenure_churn['count'],
    textposition='outside', name='Customer Count'
), row=1, col=2)

fig.update_layout(template=DARK, height=400, showlegend=False,
    title_text='⏳ Tenure Analysis — Early Months Are Critical',
    title_font=dict(size=16, color='#E2E8F0'))
fig.show()

print('\n💡 KEY INSIGHT: Customers in their first 12 months have the highest churn risk.')
print('   Retention programs should focus heavily on new customers.')

## 5. Categorical Feature Analysis <a id='5-categorical'></a>

In [ ]:
cat_features = ['contract_type', 'internet_service', 'payment_method']
fig = make_subplots(rows=1, cols=3, subplot_titles=[f.replace('_', ' ').title() for f in cat_features])

for i, feat in enumerate(cat_features):
    grp = df.groupby(feat)['churn'].agg(['mean', 'count']).reset_index()
    grp = grp.sort_values('mean', ascending=True)
    colors = ['#EF4444' if r > 0.40 else '#F59E0B' if r > 0.20 else '#22C55E' for r in grp['mean']]
    fig.add_trace(go.Bar(
        y=grp[feat], x=grp['mean'], orientation='h',
        marker_color=colors,
        text=[f'{r:.1%}' for r in grp['mean']], textposition='outside',
        showlegend=False
    ), row=1, col=i+1)

fig.update_layout(template=DARK, height=380,
    title_text='📋 Churn Rate by Categorical Features',
    title_font=dict(size=17, color='#E2E8F0'))
fig.update_xaxes(tickformat='.0%')
fig.show()

print('\n💡 KEY INSIGHTS:')
print('   Contract Type: Month-to-Month customers churn at ~3× the rate of annual/biannual contracts')
print('   Internet: Fiber Optic users show higher churn — possibly due to competitive alternatives')
print('   Payment: Electronic Check users tend to churn more (convenience barrier to switching)')

In [ ]:
pivot = df.groupby(['contract_type', 'internet_service'])['churn'].mean().unstack().fillna(0)

fig = px.imshow(
    pivot, text_auto='.1%', aspect='auto',
    color_continuous_scale=['#0d1117', '#1A56DB', '#7C3AED', '#EF4444'],
    title='🔥 Churn Rate Heatmap: Contract Type × Internet Service',
    labels={'color': 'Churn Rate'}, template=DARK
)
fig.update_layout(height=320, title_font=dict(size=16, color='#E2E8F0'))
fig.show()

print('\n💡 HIGHEST RISK SEGMENT: Month-to-Month + Fiber Optic')
print('   → Prioritize proactive outreach for this combination')

## 6. Correlation Analysis <a id='6-correlation'></a>

In [ ]:
num_df = df.select_dtypes(include=np.number).drop(columns=['tenure_bucket'], errors='ignore')
corr = num_df.corr()

fig = px.imshow(
    corr, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='🔗 Feature Correlation Matrix',
    template=DARK
)
fig.update_layout(height=480, title_font=dict(size=16, color='#E2E8F0'))
fig.show()

# Top correlations with churn
churn_corr = corr['churn'].drop('churn').sort_values(key=abs, ascending=False)
print('\n📊 FEATURES MOST CORRELATED WITH CHURN:')
for feat, val in churn_corr.items():
    direction = '↑ Positive' if val > 0 else '↓ Negative'
    print(f'   {feat:<25} {val:+.3f}  {direction}')

## 7. Customer Segments <a id='7-segments'></a>

In [ ]:
# Scatter: Monthly Charges vs Tenure — coloured by Churn
sample = df.sample(1500, random_state=42)

fig = px.scatter(
    sample, x='tenure_months', y='monthly_charges',
    color=sample['churn'].map({0: 'Retained', 1: 'Churned'}),
    color_discrete_map={'Retained': '#1A56DB', 'Churned': '#EF4444'},
    size='support_calls', size_max=14,
    opacity=0.7,
    hover_data=['satisfaction_score', 'contract_type', 'total_charges'],
    title='🧭 Customer Landscape: Tenure × Monthly Charges (size = Support Calls)',
    labels={'tenure_months': 'Tenure (Months)', 'monthly_charges': 'Monthly Charges (INR)'},
    template=DARK
)
fig.update_layout(height=480, title_font=dict(size=16, color='#E2E8F0'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

print('\n💡 INSIGHT: High-charge + low-tenure customers cluster in the churn zone.')
print('   Larger bubble = more support calls — a strong churn predictor.')

In [ ]:
sat_grp = df.groupby('satisfaction_score')['churn'].agg(['mean','count']).reset_index()
sat_grp.columns = ['score', 'churn_rate', 'count']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Churn Rate by Satisfaction', 'Customer Count'])
colors = ['#EF4444', '#F97316', '#F59E0B', '#22C55E', '#16A34A']

fig.add_trace(go.Bar(x=sat_grp['score'], y=sat_grp['churn_rate'],
    marker_color=colors,
    text=[f'{r:.1%}' for r in sat_grp['churn_rate']], textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=sat_grp['score'], y=sat_grp['count'],
    marker_color='#1A56DB',
    text=sat_grp['count'], textposition='outside'), row=1, col=2)

fig.update_layout(template=DARK, height=380, showlegend=False,
    title_text='😊 Satisfaction Score Impact on Churn',
    title_font=dict(size=16, color='#E2E8F0'))
fig.show()

## 8. Key Insights Summary <a id='8-insights'></a>

## 📌 EDA Summary — Key Findings

| # | Finding | Business Implication |
|---|---------|---------------------|
| 1 | **First 12 months** have the highest churn risk | Onboarding & early retention programmes are critical |
| 2 | **Month-to-Month contracts** churn at 3× the rate | Incentivise annual contract upgrades |
| 3 | **High monthly charges + low tenure** = highest risk segment | Target high-value new customers first |
| 4 | **Low satisfaction score (1-2)** strongly predicts churn | CSAT surveys + proactive support escalation |
| 5 | **Support calls** positively correlate with churn | Reduce friction — resolve issues on first contact |
| 6 | **Fiber Optic + Month-to-Month** = worst combination | Bundle discounts & contract incentives for this group |

---

### 🔜 Next Steps
➡️ [Notebook 02 — Feature Engineering & Model Training](./02_Model_Training_and_Evaluation.ipynb)  
➡️ [Notebook 03 — Business Impact & ROI Analysis](./03_Business_Impact_and_ROI.ipynb)